# Frequency probing: KernelSynth and Light TSMixup backgrounds

## tl;dr

This notebook repeats the pure-tone probing protocol with a known added tone
on generated backgrounds, at the same ten stages. Each of 100 backgrounds per
family creates its own labelled sweep and therefore its own SV for every
checkpoint, stage and task; the export summarises them by median and
interquartile range. Each background also carries the lock/control pair it has
in the recovery count of Appendix G, so the located rates at candidate sites
and beside them can be compared stage by stage with what the forecast returns.
Results are computed below before they are summarised.

## Context & Methods

Frozen Chronos-Bolt-Tiny checkpoints are read at ten probe points: the [REG]
vector after each of the four encoder blocks (its last position) and after each
of the four decoder blocks (the single query token), the input of the output
projection, and its output, the quantile head. The seven hierarchical binary
frequency tasks are ordered by decision threshold: LL, L, LH, Mid, HL, H and HH.
We retain their existing bands and boundaries.

**Description length by stage.** For each task, unique frequencies are assigned
to five class-stratified prequential blocks. All phases at a frequency remain in
the same block. The first block is charged uniformly at one bit per label; every
later block is encoded by a linear probe trained only on preceding blocks. The
scaler and PCA (up to 20 components) are fitted inside that preceding-data fit,
followed by logistic regression. A fixed seed of 42 governs splits, probe
randomness and both controls. The reported space saving is
\[
SV = 1 - \frac{L_{\mathrm{MDL}}}{n}.
\]
Positive SV means label compression, zero means no saving, and negative values
are retained. The initial uniform block sets a design-dependent upper bound
near 0.8. SV measures the readability of frequency labels from these
representations; it does not measure compression of waveform samples.

**Two controls.** *Shuffled labels*: the same blocks and probe on permuted
labels. *Untrained network*: the same architecture with randomly initialised
weights reads the same contexts and is scored in the same way. What the
untrained network already exposes comes from the tokenisation and the
architecture; only the gap between the two networks can be attributed to training.

**Lock/control reading.** This mirrors the recovery count of the frequency
analysis, read inside the network instead of on the forecast. A tone on a
candidate site (lock) and a tone beside it (control, more than 2 Hz from every
candidate of the geometry) are presented at every phase. A tone is *located* at
a stage when the three band probes on its path (Mid, then L or H, then one of
LL, LH, HL, HH) all return its label, which places it in an eighth of the band.
The probes are fitted on the sweep without the frequencies within 1 Hz of the
lock or control tones, so no located tone was seen in fitting. The outcome is
the located rate at lock and control and their difference, stage by stage. Task
boundaries lie beside several candidate sites, so the reading is repeated on the
pairs whose tones all lie at least 4 Hz from the boundaries on their path.

### Key assumptions

- Signals are sampled at 512 Hz. The full grid is 2–250 Hz in 1-Hz steps,
  with ten explicitly distinct phases per frequency. Candidate sites keep
  their exact, possibly non-integer, value in the lock/control reading.
- Context length is the largest \(L\le512\) satisfying \(L\bmod P=0\) and
  \((L-P)\bmod S=0\), avoiding masked patch padding and uncovered samples.
- The published checkpoint and the fifteen retrained geometries use their
  existing, identified weights; no model is retrained here.
- The grouped five-block estimator is the repository's adapted protocol.
  The figure layout follows [Pagani et al., Figure 3](https://arxiv.org/html/2605.06361v1).
- These values are not those of `02_mdl_bandtasks.parquet` (Model B), whose
  blocks are not grouped by frequency.

### Set up the experiment

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

REPO = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "support_scripts" / "probing_lib.py").is_file()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from the patchAliasing repository.")
sys.path.insert(0, str(REPO / "notebooks" / "support_scripts"))

from probing_protocol import ProbeConfig, context_length, make_design
from probing_lib import MODELS

MODE = os.environ.get("PATCHALIASING_PROBING_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("PATCHALIASING_PROBING_MODE must be 'full' or 'smoke'.")
# The untrained-network control doubles the cost of every background dataset.
# Set RANDOM_INIT = False to run the trained network alone; the pure-tone
# notebook always carries the control.
RANDOM_INIT = True
cfg = ProbeConfig.smoke(random_init=RANDOM_INIT) if MODE == "smoke" else ProbeConfig(random_init=RANDOM_INIT)
models = tuple(model for model in MODELS if model.tag == "retrained-p16-s16") if MODE == "smoke" else MODELS
ROOT = REPO / "notebooks" / "_run" / "probing"
DEVICE = "cpu"
TASK_ORDER = ["LL", "L", "LH", "Mid", "HL", "H", "HH"]
STAGE_ORDER = ["enc_0", "enc_1", "enc_2", "enc_3", "dec_0", "dec_1", "dec_2", "dec_3",
               "output_reg", "output_head"]
# Tracked copies of the values the report quotes (notebooks/_run is not in the repository).
PUBLISHED = REPO / "notebooks" / "signal_analysis" / "results" / "probing"

print(f"Mode: {MODE}; device: {DEVICE}; checkpoints: {len(models)}")
if MODE == "smoke":
    display(Markdown("**SMOKE CHECK — reduced design; these results are not report estimates.**"))

def lock_control_panel(summary, family, network="trained", suffix=""):
    """Geometry by stage: located rate at lock / at control, in per cent."""
    part = summary.loc[summary["family"].eq(family) & summary["network"].eq(network)].copy()
    part["cell"] = [("-" if pd.isna(l) or pd.isna(c) else f"{l:.0f}/{c:.0f}")
                    for l, c in zip(part[f"lock_pct{suffix}"], part[f"control_pct{suffix}"])]
    return part.pivot(index="model", columns="stage", values="cell").reindex(
        index=[model.tag for model in models], columns=STAGE_ORDER)


def net_panel(summary, family, network="trained", suffix=""):
    part = summary.loc[summary["family"].eq(family) & summary["network"].eq(network)]
    return part.pivot(index="model", columns="stage", values=f"net_pct{suffix}").reindex(
        index=[model.tag for model in models], columns=STAGE_ORDER).round(1)


def publish(paths):
    """Copy full-design exports next to the notebook so the report values are traceable."""
    import shutil
    if MODE != "full":
        return []
    PUBLISHED.mkdir(parents=True, exist_ok=True)
    copied = []
    for path in paths:
        shutil.copy2(path, PUBLISHED / path.name)
        copied.append(PUBLISHED / path.name)
    return copied

## Data

The Appendix G recipes generate 100 independent backgrounds per family.
Light TSMixup uses 1–4 components, with frequencies separated by at least
10 Hz; KernelSynth uses at most five kernels. Each canonical 544-sample draw
is generated once and normalised to unit variance, then shared across all
checkpoints and cropped to their exact context lengths.

Every background receives the same frequency/phase sweep as the pure-tone
notebook. The injected sinusoid has amplitude 1.5 times the canonical
background standard deviation; the sum is not renormalised. Its known
frequency supplies the label, including when the background itself contains
nearby energy.

One background therefore contributes a *dataset* of frequency/phase
variants, rather than a single binary compression outcome for one waveform.
The full result contains 16 × 2 × 100 × 10 × 7 = 224,000 task/stage rows.

For the lock/control reading, background *i* of a geometry receives the same
candidate site and the same control side it has in the recovery count
(Appendix G), so the two analyses see the same pairs. The lock and the control
tone are each read at the ten phases, by probes fitted on the sweep of that
same background.

In [ ]:
design = make_design(cfg)
coverage = pd.DataFrame([
    {"model": model.tag, "P": model.P, "S": model.S,
     "context_samples": context_length(model.P, model.S)}
    for model in models
])
display(coverage)
print(f"{design['frequency'].nunique()} frequencies × {cfg.n_phases} phases = {len(design)} contexts per sweep")
display(design.head(6))

print(f"Backgrounds per family: {cfg.n_backgrounds}; families: KernelSynth and Light TSMixup")
print(f"Expected task/stage rows: {len(models) * 2 * cfg.n_backgrounds * 10 * 7:,}")

## Results

### Run or resume the background sweeps

The runner saves atomic background/model shards and validates all expected
rows before exporting counts. A failed draw or fit remains an error; it is
never dropped from the denominator. Repeating the same command resumes
matching completed work. Full mode requires exactly 100 valid background
results per family, checkpoint, stage and task.

In [ ]:
from probing_lib import run_complex

results, run_dir = run_complex(ROOT, cfg=cfg, models=models, device=DEVICE)
trials = pd.read_csv(run_dir / "lock_control.csv")
prefix = "smoke_" if MODE == "smoke" else ""
lock_control = pd.read_csv(run_dir / f"{prefix}complex_lock_control.csv")
print(f"Validated SV rows: {len(results):,}; lock/control rows: {len(trials):,}")
print(f"Run directory: {run_dir}")

### Output-head SV across backgrounds, for Section 5.2

Rows follow the geometry comparison of Table 5; columns are the seven
hierarchical tasks. A count of backgrounds with SV > 0 saturates near 100 and
says little, so the table reports the median across backgrounds with the
quartiles; the count stays in the CSV export.

In [ ]:
head = results.loc[results["stage"].eq("output_head")].copy()
keys = ["model", "P", "S", "official", "family", "task"]
if head.duplicated(keys + ["background_id"]).any():
    raise ValueError("Duplicate background outcomes cannot enter the table.")
expected_ids = set(range(cfg.n_backgrounds))
if not head.groupby(keys)["background_id"].agg(lambda values: set(values) == expected_ids).all():
    raise ValueError("A table cell is missing one or more required backgrounds.")
counts = head.groupby(keys, as_index=False).agg(
    compresses=("SV", lambda values: int((values > 0).sum())), n=("background_id", "size"),
    median=("SV", "median"), q25=("SV", lambda values: values.quantile(.25)),
    q75=("SV", lambda values: values.quantile(.75)))
counts["cell"] = [f"{m:.2f} [{a:.2f}, {b:.2f}]" for m, a, b in zip(counts["median"], counts["q25"], counts["q75"])]
model_order = [model.tag for model in models]
for family, label in (("tsmixup", "Light TSMixup"), ("kernelsynth", "KernelSynth")):
    display(Markdown(f"**{label}: median SV [quartiles] over {cfg.n_backgrounds} backgrounds ({MODE})**"))
    panel = counts.loc[counts["family"].eq(family)].pivot(index="model", columns="task", values="cell")
    display(panel.reindex(index=model_order, columns=TASK_ORDER))

### Continuous SV distributions by stage

The following reference-checkpoint table reports the median and interquartile
range across backgrounds at all ten stages, together with the medians of the
two controls. These are descriptive spreads, not confidence intervals.

In [ ]:
reference = results.loc[results["model"].eq("retrained-p16-s16")]
aggregations = dict(
    median_sv=("SV", "median"),
    q25_sv=("SV", lambda values: values.quantile(.25)),
    q75_sv=("SV", lambda values: values.quantile(.75)),
    median_shuffled_sv=("SV_shuffled", "median"))
if "SV_random_init" in reference:
    aggregations["median_random_init_sv"] = ("SV_random_init", "median")
distribution = reference.groupby(["family", "stage", "task"], sort=False).agg(**aggregations)
display(distribution.round(3))

In [ ]:
figure_paths = sorted(run_dir.glob(f"{prefix}sv_distribution_p16_s16.png"))
if len(figure_paths) != 1:
    raise RuntimeError(f"Expected one reference distribution figure, found {len(figure_paths)}.")
display(Image(filename=str(figure_paths[0]), width=1100))

### Lock/control reading

For each generator, a cell is the percentage of contexts in which the added
tone is located when it sits on a candidate site / when it sits beside it,
over the backgrounds of the design. The second table of each pair is the
difference in percentage points, and the third repeats it away from the task
boundaries. Read them beside Table 5 of the report: there the forecast returns
a tone more often on a site than beside it; here the question is whether the
internal state places the tone equally well in the two cases.

In [ ]:
for family, label in (("tsmixup", "Light TSMixup"), ("kernelsynth", "KernelSynth")):
    display(Markdown(f"**{label}**"))
    display(lock_control_panel(lock_control, family).rename_axis("lock/control located, %"))
    display(net_panel(lock_control, family).rename_axis("net (lock - control), points"))
    display(net_panel(lock_control, family, suffix="_off").rename_axis("net, off-boundary pairs"))
    if lock_control["network"].eq("random_init").any():
        display(net_panel(lock_control, family, network="random_init").rename_axis("net, untrained network"))

### Export inventory

In [ ]:
exported = sorted(path for path in run_dir.rglob("*") if path.is_file() and path.suffix in {".png", ".pdf", ".tex", ".txt"})
print(f"{len(exported)} table/figure/caption export files; raw outcomes and distribution summaries are saved as CSV.")
display(pd.DataFrame({"export": [str(path.relative_to(run_dir)) for path in exported]}).head(24))
copied = publish([path for path in run_dir.iterdir() if path.is_file() and path.suffix in {".tex", ".pdf", ".txt"}]
                 + [run_dir / name for name in ("sv_distribution.csv", "compression_counts.csv", "complex_lock_control.csv")
                    if (run_dir / name).is_file()])
print(f"{len(copied)} files copied to {PUBLISHED}" if copied else "Smoke mode: nothing copied to the tracked folder.")

## Takeaways

Read the medians together with their quartiles and the two controls, and the
lock/control differences next to the recovery count. A readable band at the
output head does not by itself establish recovery in the forecast, and a
difference at a site whose pairs lie beside a task boundary is not evidence of
a site effect. The solar series is not used in this experiment.

In [ ]:
if MODE == "smoke":
    display(Markdown("**Smoke validation only:** the reduced-background values cannot be used as the report's 100-background tables."))
else:
    if cfg.n_backgrounds != 100 or not counts["n"].eq(100).all():
        raise ValueError("The report tables require 100 valid outcomes in every cell.")
    display(Markdown("Every displayed full-design value rests on 100 valid background outcomes. The LaTeX/CSV exports are ready for Section 5.2 and Appendix G."))